# Q-Mini training pipeline
From-scratch, attention-free CausalConv-MoE model. No pretrained weights are downloaded. Keep datasets and checkpoints in Drive, not this public source repository.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DATA = Path('/content/drive/MyDrive/QMini-Light-0.7M/datasets')
ART = Path('/content/drive/MyDrive/QMini-Light-0.7M/artifacts')
ART.mkdir(parents=True, exist_ok=True)
print('Data:', DATA, '| Artifacts:', ART)


In [ ]:
import os, subprocess
REPO = Path('/content/Q-Mini')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/lion36508/Q-mini-light-0.7M.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
os.chdir(REPO)

# Keep Colab's preinstalled CUDA-enabled PyTorch build. Installing the project
# with normal dependency resolution can replace it with a different wheel.
subprocess.run([
    'python','-m','pip','install',
    'numpy>=1.24','tokenizers>=0.15','pyarrow>=15.0','pytest>=8.0','packaging>=23',
],check=True)
subprocess.run([
    'python','-m','pip','install','--no-deps','-e','.'
],check=True)

import torch
from packaging.version import Version
torch_version = Version(torch.__version__.split('+',1)[0])
if torch_version < Version('2.10.0'):
    raise RuntimeError(
        f'Q-Mini requires PyTorch >=2.10.0, but Colab has {torch.__version__}. '
        'Select a compatible Colab runtime; do not blindly replace its CUDA wheel.'
    )
print('PyTorch',torch.__version__,'CUDA build',torch.version.cuda)
print('CUDA available',torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU',torch.cuda.get_device_name(0))
    free_bytes,total_bytes = torch.cuda.mem_get_info(0)
    print(f'GPU memory free: {free_bytes/1024**3:.2f} GiB / total: {total_bytes/1024**3:.2f} GiB')
else:
    print('Only the small smoke stage may use CPU. The 0.7M/0.7M GPU stages are blocked unless CUDA is available.')


In [ ]:
# Verify the empty Q-Mini architecture before loading any corpus.
# This performs no optimizer step, does not alter weights, and does not open the Drive datasets.
# Preflight backward only checks finite gradients; it never calls optimizer.step().
# Use a full research-profile preflight on GPU. On CPU-only runtimes, use the
# tiny smoke config so the notebook can still reach its explicitly CPU-safe
# smoke stage instead of failing here before data preparation.
if torch.cuda.is_available():
    PREFLIGHT_CONFIG = REPO/'configs'/'qmini_research_0.7m.json'
    PREFLIGHT_DEVICE = 'cuda'
    PREFLIGHT_MAX_PARAMETERS = '1000000'
else:
    PREFLIGHT_CONFIG = REPO/'configs'/'qmini_smoke.json'
    PREFLIGHT_DEVICE = 'cpu'
    PREFLIGHT_MAX_PARAMETERS = '1000000'

ARCHITECTURE_PREFLIGHT = [
    'python','-m','qmini.cli','preflight',
    '--config',str(PREFLIGHT_CONFIG),
    '--device',PREFLIGHT_DEVICE,
    '--sequence-length','8',
    '--max-parameters',PREFLIGHT_MAX_PARAMETERS,
]
subprocess.run(ARCHITECTURE_PREFLIGHT,check=True)
print('Architecture preflight complete: no corpus loaded and no weights updated.')



In [ ]:
TOK = ART/'tokenizer_light_0.7m'/'tokenizer.json'; CFG = ART/'qmini_light_0.7m_active_config.json'
TOK.parent.mkdir(parents=True, exist_ok=True)
if not TOK.exists():
    subprocess.run([
        'python','-m','qmini.cli','tokenizer',
        '--data-dir',str(DATA),'--output',str(TOK),
        '--vocab-size','4096','--base-config',str(REPO/'configs'/'qmini_research_0.7m.json'),
        '--valid-fraction','0.02',
        '--config-output',str(CFG)
    ],check=True)
else:
    print('Existing tokenizer preserved:', TOK)
if not CFG.is_file():
    raise FileNotFoundError(
        'Tokenizer exists but its active config is missing. Restore the matching config; '
        'do not silently retrain or overwrite the tokenizer.'
    )

# Create a genuinely small smoke model but retain the exact tokenizer vocabulary.
# The stock smoke config has a 128-token vocabulary and must never be paired with
# the full-corpus token streams.
from qmini.config import QMiniConfig
base_config = QMiniConfig.from_json(CFG)
smoke_values = base_config.to_dict()
smoke_values.update({
    'd_model': 32,
    'd_hidden': 48,
    'n_layers': 2,
    'n_experts': 2,
    'top_k': 1,
    'max_context': 64,
    'kernel_size': 5,
    'dilations': [1, 2],
    'max_thinking_steps': 2,
    'thinking_steps': 1,
})
smoke_config = QMiniConfig.from_dict(smoke_values)
SMOKE_CFG = ART/'qmini_smoke_active_config.json'
smoke_config.save(SMOKE_CFG)
print('Base vocabulary:', base_config.vocab_size)
print('Smoke vocabulary:', smoke_config.vocab_size)
print('Smoke model parameters are computed by the runtime, not hard-coded.')


In [ ]:
TOKENS = ART/'tokens_light_0.7m_v1'
required_artifacts = [
    TOKENS/'train.bin',
    TOKENS/'valid.bin',
    TOKENS/'train.mask.bin',
    TOKENS/'valid.mask.bin',
    TOKENS/'dataset.json',
]
present = [item.is_file() for item in required_artifacts]
if any(present) and not all(present):
    raise RuntimeError(
        'An incomplete token artifact set already exists. It has been preserved. '
        'Set TOKENS to a new versioned directory such as tokens_v2; do not mix old and new streams.'
    )
if not all(present):
    subprocess.run([
        'python','-m','qmini.cli','prepare',
        '--data-dir',str(DATA),'--tokenizer',str(TOK),
        '--output-dir',str(TOKENS),'--valid-fraction','0.02',
        '--loss-mask','auto'
    ],check=True)
else:
    print('Existing token streams preserved:', TOKENS)

# Verify that token IDs, loss masks and tokenizer all belong to one preparation run.
import hashlib, json
manifest = json.loads((TOKENS/'dataset.json').read_text(encoding='utf-8'))
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
if manifest.get('tokenizer_sha256') != sha256_file(TOK):
    raise RuntimeError('Tokenizer hash does not match token-stream manifest.')
for stream_name, mask_name, manifest_key, mask_key in [
    ('train.bin','train.mask.bin','train_tokens_sha256','train_mask_sha256'),
    ('valid.bin','valid.mask.bin','valid_tokens_sha256','valid_mask_sha256'),
]:
    stream = TOKENS/stream_name
    mask = TOKENS/mask_name
    if mask.stat().st_size != stream.stat().st_size // 2:
        raise RuntimeError(f'Loss-mask length mismatch for {stream_name}.')
    if manifest.get(manifest_key) != sha256_file(stream):
        raise RuntimeError(f'{stream_name} hash does not match dataset manifest.')
    if manifest.get(mask_key) != sha256_file(mask):
        raise RuntimeError(f'{mask_name} hash does not match dataset manifest.')
print('train tokens:', (TOKENS/'train.bin').stat().st_size//2)
print('valid tokens:', (TOKENS/'valid.bin').stat().st_size//2)
print('train documents:', manifest['train_documents'])
print('valid documents:', manifest['valid_documents'])
print('loss-mask mode:', manifest['loss_mask_mode'])


In [ ]:
# A short, restartable learning proof comes before every expensive run.
# This smoke model uses the same tokenizer but is not the 0.7M/5B target model.
import hashlib, json, os, subprocess, torch
from qmini.config import QMiniConfig
from qmini.evaluate import evaluate_language_model
from qmini.model import QMiniMoE

SMOKE_STEPS = 50
SMOKE_BATCH_SIZE = 1
SMOKE_SEQ_LEN = 32
SMOKE_GRAD_ACCUM = 1
SMOKE_EVAL_BATCHES = 8
LEARNING_EVAL_SEED = 20261109
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SOURCE_COMMIT = subprocess.run(
    ['git','rev-parse','HEAD'], capture_output=True, text=True, check=True
).stdout.strip()
implementation_files = [
    REPO/'qmini'/'model.py',
    REPO/'qmini'/'train.py',
    REPO/'qmini'/'data.py',
    REPO/'qmini'/'quantization.py',
    REPO/'qmini'/'tokenizer.py',
    REPO/'qmini'/'config.py',
    REPO/'qmini'/'generate.py',
    REPO/'qmini'/'evaluate.py',
    REPO/'qmini'/'export.py',
    REPO/'qmini'/'packed_runtime.py',
]
implementation_hash = hashlib.sha256()
for source_file in implementation_files:
    implementation_hash.update(source_file.relative_to(REPO).as_posix().encode('utf-8'))
    implementation_hash.update(source_file.read_bytes())
IMPLEMENTATION_SHA256 = implementation_hash.hexdigest()

smoke_config = QMiniConfig.from_json(SMOKE_CFG)
smoke_identity = json.dumps({
    'source_commit': SOURCE_COMMIT,
    'implementation_sha256': IMPLEMENTATION_SHA256,
    'corpus_sha256': manifest['corpus_sha256'],
    'tokenizer_sha256': manifest['tokenizer_sha256'],
    'config': smoke_config.to_dict(),
    'training': {
        'steps': SMOKE_STEPS, 'batch_size': SMOKE_BATCH_SIZE,
        'seq_len': SMOKE_SEQ_LEN, 'grad_accum': SMOKE_GRAD_ACCUM,
        'learning_rate': 3e-4, 'weight_decay': 0.1,
        'seed': smoke_config.seed, 'eval_seed': LEARNING_EVAL_SEED,
    },
}, sort_keys=True).encode('utf-8')
SMOKE_KEY = hashlib.sha256(smoke_identity).hexdigest()[:16]
SMOKE = ART/'runs'/f'smoke_{SMOKE_KEY}'
SMOKE.mkdir(parents=True, exist_ok=True)

# Fixed held-out windows and seed are used before and after training.
torch.manual_seed(smoke_config.seed)
random_model = QMiniMoE(smoke_config).to(DEVICE).eval()
before_report = evaluate_language_model(
    random_model, TOKENS/'valid.bin', TOKENS/'valid.mask.bin',
    batch_size=1, seq_len=SMOKE_SEQ_LEN, batches=SMOKE_EVAL_BATCHES,
    seed=LEARNING_EVAL_SEED,
)
del random_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

smoke_args = [
    'python','-m','qmini.cli','train',
    '--config',str(SMOKE_CFG),
    '--train-bin',str(TOKENS/'train.bin'),
    '--valid-bin',str(TOKENS/'valid.bin'),
    '--train-mask-bin',str(TOKENS/'train.mask.bin'),
    '--valid-mask-bin',str(TOKENS/'valid.mask.bin'),
    '--tokenizer',str(TOK),
    '--output-dir',str(SMOKE),
    '--steps',str(SMOKE_STEPS),
    '--batch-size',str(SMOKE_BATCH_SIZE),
    '--seq-len',str(SMOKE_SEQ_LEN),
    '--grad-accum',str(SMOKE_GRAD_ACCUM),
    '--eval-every','10','--save-every','10',
    '--eval-batches',str(SMOKE_EVAL_BATCHES),
]
checkpoint_path = SMOKE/'latest.pt'
if checkpoint_path.is_file():
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    if checkpoint.get('config') != smoke_config.to_dict():
        raise RuntimeError('Existing smoke checkpoint config differs; it was preserved.')
    checkpoint_step = int(checkpoint.get('step', 0))
    if checkpoint_step < SMOKE_STEPS:
        smoke_args.append('--resume')
        subprocess.run(smoke_args, check=True)
    elif not (SMOKE/'final_model'/'model_manifest.json').is_file():
        smoke_args[smoke_args.index('--steps') + 1] = str(checkpoint_step + 1)
        smoke_args.extend(['--resume','--extend-schedule'])
        subprocess.run(smoke_args, check=True)
else:
    subprocess.run(smoke_args, check=True)

if not checkpoint_path.is_file():
    raise RuntimeError(
        f'Smoke training did not create the expected resumable checkpoint: {checkpoint_path}'
    )
checkpoint_payload = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
if checkpoint_payload.get('format') != 'qmini-checkpoint-v2':
    raise RuntimeError('Smoke run did not produce a resumable Q-Mini checkpoint.')
if checkpoint_payload.get('config') != smoke_config.to_dict():
    raise RuntimeError('Smoke checkpoint config differs from the selected model config.')
trained_state = checkpoint_payload.get('model')
if not isinstance(trained_state, dict) or not trained_state:
    raise RuntimeError('Smoke checkpoint does not contain a non-empty model state.')
trained_model = QMiniMoE(smoke_config).to(DEVICE).eval()
trained_model.load_state_dict(trained_state, strict=True)
after_report = evaluate_language_model(
    trained_model, TOKENS/'valid.bin', TOKENS/'valid.mask.bin',
    batch_size=1, seq_len=SMOKE_SEQ_LEN, batches=SMOKE_EVAL_BATCHES,
    seed=LEARNING_EVAL_SEED,
)
del trained_model, trained_state
if torch.cuda.is_available():
    torch.cuda.empty_cache()

learning_proof = {
    'format': 'qmini-learning-proof-v1',
    'source_commit': SOURCE_COMMIT,
    'implementation_sha256': IMPLEMENTATION_SHA256,
    'corpus_sha256': manifest['corpus_sha256'],
    'tokenizer_sha256': manifest['tokenizer_sha256'],
    'smoke_config': smoke_config.to_dict(),
    'eval_seed': LEARNING_EVAL_SEED,
    'before_random_initialization': before_report,
    'after_smoke_training': after_report,
    'minimum_nll_improvement': 0.05,
    'measured_nll_improvement': float(
        before_report['cross_entropy'] - after_report['cross_entropy']
    ),
    'heldout_nll_improved': (
        before_report['cross_entropy'] - after_report['cross_entropy'] >= 0.05
    ),
}
proof_path = ART/'reports'/f'learning_proof_{SMOKE_KEY}.json'
proof_path.parent.mkdir(parents=True, exist_ok=True)
proof_path.write_text(json.dumps(learning_proof, indent=2), encoding='utf-8')
LEARNING_PROOF_PASSED = bool(learning_proof['heldout_nll_improved'])
print('Held-out NLL before:', before_report['cross_entropy'])
print('Held-out NLL after :', after_report['cross_entropy'])
print('Held-out NLL improvement:', learning_proof['measured_nll_improvement'])
print('Learning proof:', 'PASSED' if LEARNING_PROOF_PASSED else 'NOT PROVEN')
print('Proof report:', proof_path)


## Stage 1 — optional 0.7M intermediate scaling experiment

This stage is disabled by default and is available only after the small fixed-seed learning proof passes. It uses the 0.7M configuration with a short, configurable token/step budget and its own hash-keyed Drive checkpoint. This provides an intermediate scaling rung before the optional 0.7M research run; it does not establish 5B training readiness.


In [ ]:
# Stage 1: optional intermediate 0.7M experiment after the learning proof.
# This is a small scaling rung, not the 0.7M research run or the 5B target.
RUN_INTERMEDIATE_LIGHT = False
INTERMEDIATE_LIGHT_STEPS = 100
INTERMEDIATE_LIGHT_BATCH = 1
INTERMEDIATE_LIGHT_SEQ = 128
INTERMEDIATE_LIGHT_GRAD_ACCUM = 1
INTERMEDIATE_LIGHT_EVAL_EVERY = 25
INTERMEDIATE_LIGHT_SAVE_EVERY = 25
INTERMEDIATE_LIGHT_EVAL_BATCHES = 4
INTERMEDIATE_LIGHT_LR = 3e-4
INTERMEDIATE_LIGHT_WEIGHT_DECAY = 0.1

if not LEARNING_PROOF_PASSED:
    print('0.7M scaling stage is blocked because the fixed held-out learning proof did not pass.')
elif not RUN_INTERMEDIATE_LIGHT:
    print('0.7M intermediate training is disabled. Set RUN_INTERMEDIATE_LIGHT=True after reviewing the budget.')
else:
    if not torch.cuda.is_available():
        raise RuntimeError(
            'The 0.7M intermediate stage requires a CUDA GPU. In Colab select '
            'Runtime > Change runtime type > GPU, reconnect, and rerun the setup cells.'
        )
    print('0.7M training GPU:', torch.cuda.get_device_name(0))
    config_light_path = REPO/'configs'/'qmini_5m.json'
    config_light = QMiniConfig.from_json(config_light_path)
    if config_light.vocab_size != smoke_config.vocab_size:
        raise RuntimeError('0.7M and prepared tokenizer vocabularies do not match.')
    if config_light.estimated_parameter_count() > 800_000:
        raise RuntimeError('Unexpected parameter count for the 0.7M intermediate profile.')
    if INTERMEDIATE_LIGHT_SEQ > config_light.max_context:
        raise RuntimeError('0.7M sequence length exceeds configured max_context.')

    identity_5m = {
        'source_commit': SOURCE_COMMIT,
        'implementation_sha256': IMPLEMENTATION_SHA256,
        'corpus_sha256': manifest['corpus_sha256'],
        'train_sha256': manifest['train_tokens_sha256'],
        'valid_sha256': manifest['valid_tokens_sha256'],
        'tokenizer_sha256': manifest['tokenizer_sha256'],
        'config': config_light.to_dict(),
        'batch': INTERMEDIATE_LIGHT_BATCH,
        'sequence_length': INTERMEDIATE_LIGHT_SEQ,
        'gradient_accumulation': INTERMEDIATE_LIGHT_GRAD_ACCUM,
        'learning_rate': INTERMEDIATE_LIGHT_LR,
        'weight_decay': INTERMEDIATE_LIGHT_WEIGHT_DECAY,
        'eval_every': INTERMEDIATE_LIGHT_EVAL_EVERY,
        'save_every': INTERMEDIATE_LIGHT_SAVE_EVERY,
        'eval_batches': INTERMEDIATE_LIGHT_EVAL_BATCHES,
        'seed': config_light.seed,
    }
    KEY_LIGHT = hashlib.sha256(
        json.dumps(identity_5m, sort_keys=True).encode('utf-8')
    ).hexdigest()[:16]
    RUN_LIGHT = ART/'runs'/f'light_0_7m_{KEY_LIGHT}'
    RUN_LIGHT.mkdir(parents=True, exist_ok=True)
    ARGS_LIGHT = [
        'python','-m','qmini.cli','train',
        '--config',str(config_light_path),
        '--train-bin',str(TOKENS/'train.bin'),
        '--valid-bin',str(TOKENS/'valid.bin'),
        '--train-mask-bin',str(TOKENS/'train.mask.bin'),
        '--valid-mask-bin',str(TOKENS/'valid.mask.bin'),
        '--tokenizer',str(TOK),
        '--output-dir',str(RUN_LIGHT),
        '--steps',str(INTERMEDIATE_LIGHT_STEPS),
        '--batch-size',str(INTERMEDIATE_LIGHT_BATCH),
        '--seq-len',str(INTERMEDIATE_LIGHT_SEQ),
        '--grad-accum',str(INTERMEDIATE_LIGHT_GRAD_ACCUM),
        '--lr',str(INTERMEDIATE_LIGHT_LR),
        '--weight-decay',str(INTERMEDIATE_LIGHT_WEIGHT_DECAY),
        '--eval-every',str(INTERMEDIATE_LIGHT_EVAL_EVERY),
        '--save-every',str(INTERMEDIATE_LIGHT_SAVE_EVERY),
        '--eval-batches',str(INTERMEDIATE_LIGHT_EVAL_BATCHES),
    ]
    checkpoint_light = RUN_LIGHT/'latest.pt'
    final_manifest_light = RUN_LIGHT/'final_model'/'model_manifest.json'
    if checkpoint_light.is_file():
        state_light = torch.load(checkpoint_light, map_location='cpu', weights_only=True)
        if state_light.get('config') != config_light.to_dict():
            raise RuntimeError('Existing 0.7M checkpoint config differs; it was preserved.')
        run_json_light = RUN_LIGHT/'run.json'
        if not run_json_light.is_file():
            raise RuntimeError('Existing 0.7M checkpoint has no run manifest; it was preserved.')
        run_record_light = json.loads(run_json_light.read_text(encoding='utf-8'))
        contract_light = run_record_light.get('resume_contract', {})
        if contract_light.get('train', {}).get('sha256') != manifest['train_tokens_sha256']:
            raise RuntimeError('Existing 0.7M run uses different training tokens.')
        if contract_light.get('tokenizer', {}).get('sha256') != manifest['tokenizer_sha256']:
            raise RuntimeError('Existing 0.7M run uses a different tokenizer.')
        step_light = int(state_light.get('step', 0))
        if step_light < INTERMEDIATE_LIGHT_STEPS:
            ARGS_LIGHT.append('--resume')
            saved_schedule_light = int(state_light.get('schedule_total_steps', INTERMEDIATE_LIGHT_STEPS))
            if INTERMEDIATE_LIGHT_STEPS > saved_schedule_light:
                ARGS_LIGHT.append('--extend-schedule')
            subprocess.run(ARGS_LIGHT, check=True)
        elif not final_manifest_light.is_file():
            ARGS_LIGHT[ARGS_LIGHT.index('--steps') + 1] = str(step_light + 1)
            ARGS_LIGHT.extend(['--resume','--extend-schedule'])
            subprocess.run(ARGS_LIGHT, check=True)
        else:
            print('0.7M intermediate run already exported; checkpoint preserved:', checkpoint_light)
    else:
        subprocess.run(ARGS_LIGHT, check=True)

    print('0.7M intermediate model:', RUN_LIGHT/'final_model')
    print('0.7M intermediate manifest:', final_manifest_light)


## Training stages and safety gates

The default path is a 50-step small-model smoke/learning-proof run using a fixed held-out seed. An optional 0.7M intermediate experiment is available only after that proof passes. The 0.7M research run is separately opt-in and Drive-resumable. Its total optimizer-step target is user-controlled; there is no arbitrary 2,000-step cap. Optional assistant-only SFT is another separate stage. The 5B configuration is not a Colab training target.


In [ ]:
# Stage 2: optional long research training. Nothing expensive runs by default.
# Change RUN_LONG_TRAINING only after reviewing the resource report below.
RUN_LONG_TRAINING = False
LONG_OPTIMIZER_STEPS_OVERRIDE = None  # e.g. 50_000; None uses the documented token-budget heuristic. This is a total target, not an extra-step count.
LONG_TARGET_TOKENS_OVERRIDE = None  # e.g. 100_000_000; used only when the step override above is None.
LONG_BATCH_SIZE = 1
LONG_SEQ_LEN = min(base_config.max_context, 2048)
LONG_GRAD_ACCUM = 16
LONG_EVAL_EVERY = 100
LONG_SAVE_EVERY = 100
LONG_EVAL_BATCHES = 8
LONG_LEARNING_RATE = 3e-4
LONG_WEIGHT_DECAY = 0.1
LONG_MIN_FREE_GPU_GIB = 8.0  # conservative launch floor, not a no-OOM guarantee

if not LEARNING_PROOF_PASSED:
    print('Long training is blocked: smoke training did not improve fixed held-out NLL.')
    print('Inspect the learning-proof report and repair data/configuration before scaling.')
elif not RUN_LONG_TRAINING:
    print('Long 0.7M training is intentionally disabled.')
    print('Set RUN_LONG_TRAINING = True only after reviewing this cell and the resource budget.')
else:
    import math
    if not torch.cuda.is_available():
        raise RuntimeError(
            'The 0.7M research run requires a CUDA GPU. Select a GPU runtime in Colab; '
            'CPU fallback is intentionally refused for this long stage.'
        )
    if (
        isinstance(LONG_MIN_FREE_GPU_GIB, bool)
        or not isinstance(LONG_MIN_FREE_GPU_GIB, (int, float))
        or not math.isfinite(float(LONG_MIN_FREE_GPU_GIB))
        or LONG_MIN_FREE_GPU_GIB <= 0
    ):
        raise ValueError('LONG_MIN_FREE_GPU_GIB must be a finite positive number.')
    gpu_name = torch.cuda.get_device_name(0)
    free_gpu_bytes, total_gpu_bytes = torch.cuda.mem_get_info(0)
    required_free_gpu_bytes = int(float(LONG_MIN_FREE_GPU_GIB) * (1024 ** 3))
    print('0.7M training GPU:', gpu_name)
    print(
        f'GPU memory available before model allocation: '
        f'{free_gpu_bytes / 1024**3:.2f} GiB free / {total_gpu_bytes / 1024**3:.2f} GiB total'
    )
    if free_gpu_bytes < required_free_gpu_bytes:
        raise RuntimeError(
            f'Long training blocked: only {free_gpu_bytes / 1024**3:.2f} GiB is free; '
            f'the configured launch floor is {LONG_MIN_FREE_GPU_GIB:.2f} GiB. '
            'Close other GPU workloads or use a larger GPU. Lowering this floor does not '
            'guarantee that the run will fit.'
        )
    print(
        'GPU memory check passed. This is only a launch preflight: activation peaks, '
        'allocator fragmentation and runtime overhead can still cause OOM.'
    )
    TOKENS_PER_OPTIMIZER_STEP = LONG_BATCH_SIZE * LONG_SEQ_LEN * LONG_GRAD_ACCUM
    HEURISTIC_TARGET_TOKENS = base_config.reference_tokens_at_20_per_active_parameter()
    if LONG_TARGET_TOKENS_OVERRIDE is not None and (
        isinstance(LONG_TARGET_TOKENS_OVERRIDE, bool)
        or not isinstance(LONG_TARGET_TOKENS_OVERRIDE, int)
        or LONG_TARGET_TOKENS_OVERRIDE < 1
    ):
        raise ValueError('LONG_TARGET_TOKENS_OVERRIDE must be a positive integer or None.')
    TARGET_TOKENS = (
        LONG_TARGET_TOKENS_OVERRIDE
        if LONG_TARGET_TOKENS_OVERRIDE is not None
        else HEURISTIC_TARGET_TOKENS
    )
    REQUESTED_STEPS = (TARGET_TOKENS + TOKENS_PER_OPTIMIZER_STEP - 1) // TOKENS_PER_OPTIMIZER_STEP
    if LONG_OPTIMIZER_STEPS_OVERRIDE is None:
        if TARGET_TOKENS < TOKENS_PER_OPTIMIZER_STEP:
            raise ValueError('LONG_TARGET_TOKENS_OVERRIDE must cover at least one optimizer update.')
        BASE_STEPS = REQUESTED_STEPS
        STEP_BUDGET_SOURCE = 'documented token-budget heuristic'
    else:
        if (
            isinstance(LONG_OPTIMIZER_STEPS_OVERRIDE, bool)
            or not isinstance(LONG_OPTIMIZER_STEPS_OVERRIDE, int)
            or LONG_OPTIMIZER_STEPS_OVERRIDE < 1
        ):
            raise ValueError('LONG_OPTIMIZER_STEPS_OVERRIDE must be a positive integer or None.')
        BASE_STEPS = LONG_OPTIMIZER_STEPS_OVERRIDE
        STEP_BUDGET_SOURCE = 'user-selected total optimizer steps'
    ACTUAL_TOKENS = BASE_STEPS * TOKENS_PER_OPTIMIZER_STEP
    TRAIN_STREAM_TOKENS = int(manifest['train_tokens'])
    VALID_STREAM_TOKENS = int(manifest['valid_tokens'])
    if TRAIN_STREAM_TOKENS < LONG_SEQ_LEN + 2 or VALID_STREAM_TOKENS < LONG_SEQ_LEN + 2:
        raise RuntimeError('Prepared streams are too short for the selected long-run sequence length.')

    training_state_bytes = base_config.estimated_adamw_training_state_bytes()
    print('PROFILE: hybrid research 0.7M; this is not the 5B configuration.')
    print('Total parameters:', base_config.estimated_parameter_count())
    print('Unique active parameters/token:', base_config.estimated_unique_active_parameter_count())
    print('FP32 params+grads+AdamW state lower bound (bytes):', training_state_bytes)
    print('Per-device total RAM:', torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else 'CPU runtime')
    print('Tokens/update:', TOKENS_PER_OPTIMIZER_STEP)
    print('Heuristic token target (planning reference):', TARGET_TOKENS)
    print('Optimizer-step target:', BASE_STEPS, '| selection:', STEP_BUDGET_SOURCE)
    print('Actual tokens at selected step target:', ACTUAL_TOKENS)
    print('No arbitrary maximum optimizer-step cap is applied.')
    print('Approximate corpus passes:', round(ACTUAL_TOKENS / TRAIN_STREAM_TOKENS, 3))
    print('Validation tokens:', VALID_STREAM_TOKENS)
    print('WARNING: memory excludes activations, workspaces, fragmentation and runtime overhead.')
    print('Colab GPU availability and session duration are not guaranteed.')

    base_identity = json.dumps({
        'source_commit': SOURCE_COMMIT,
        'implementation_sha256': IMPLEMENTATION_SHA256,
        'corpus_sha256': manifest['corpus_sha256'],
        'tokenizer_sha256': manifest['tokenizer_sha256'],
        'config': base_config.to_dict(),
        'training': {
            'batch_size': LONG_BATCH_SIZE,
            'seq_len': LONG_SEQ_LEN, 'grad_accum': LONG_GRAD_ACCUM,
            'learning_rate': LONG_LEARNING_RATE,
            'weight_decay': LONG_WEIGHT_DECAY,
            'seed': base_config.seed,
            'eval_every': LONG_EVAL_EVERY, 'save_every': LONG_SAVE_EVERY,
        },
    }, sort_keys=True).encode('utf-8')
    BASE_KEY = hashlib.sha256(base_identity).hexdigest()[:16]
    BASE = ART/'runs'/f'base_{BASE_KEY}'
    args = [
        'python','-m','qmini.cli','train',
        '--config',str(CFG),
        '--train-bin',str(TOKENS/'train.bin'),
        '--valid-bin',str(TOKENS/'valid.bin'),
        '--train-mask-bin',str(TOKENS/'train.mask.bin'),
        '--valid-mask-bin',str(TOKENS/'valid.mask.bin'),
        '--tokenizer',str(TOK),
        '--output-dir',str(BASE),
        '--steps',str(BASE_STEPS),
        '--batch-size',str(LONG_BATCH_SIZE),
        '--seq-len',str(LONG_SEQ_LEN),
        '--grad-accum',str(LONG_GRAD_ACCUM),
        '--lr',str(LONG_LEARNING_RATE),
        '--weight-decay',str(LONG_WEIGHT_DECAY),
        '--eval-every',str(LONG_EVAL_EVERY),
        '--save-every',str(LONG_SAVE_EVERY),
        '--eval-batches',str(LONG_EVAL_BATCHES),
    ]
    checkpoint_path = BASE/'latest.pt'
    if checkpoint_path.is_file():
        checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
        run_manifest_path = BASE/'run.json'
        if not run_manifest_path.is_file():
            raise RuntimeError('Existing checkpoint has no run.json contract; it was preserved.')
        run_manifest = json.loads(run_manifest_path.read_text(encoding='utf-8'))
        if checkpoint.get('config') != base_config.to_dict():
            raise RuntimeError('Existing long-run checkpoint config differs; it was preserved.')
        if run_manifest.get('resume_contract', {}).get('train', {}).get('sha256') != manifest['train_tokens_sha256']:
            raise RuntimeError('Existing long-run checkpoint uses different training tokens.')
        checkpoint_step = int(checkpoint.get('step', 0))
        if checkpoint_step < BASE_STEPS:
            args.append('--resume')
            saved_schedule_steps = int(checkpoint.get('schedule_total_steps', BASE_STEPS))
            if BASE_STEPS > saved_schedule_steps:
                args.append('--extend-schedule')
            subprocess.run(args, check=True)
        elif not (BASE/'final_model'/'model_manifest.json').is_file():
            args[args.index('--steps') + 1] = str(checkpoint_step + 1)
            args.extend(['--resume','--extend-schedule'])
            subprocess.run(args, check=True)
        else:
            print('Long-run target already exported; Drive checkpoint preserved:', checkpoint_path)
    else:
        subprocess.run(args, check=True)

    EVAL_REPORT = ART/'reports'/f'language_eval_{BASE_KEY}.json'
    subprocess.run([
        'python','-m','qmini.cli','eval-lm',
        '--model-dir',str(BASE/'final_model'),
        '--tokens',str(TOKENS/'valid.bin'),
        '--mask',str(TOKENS/'valid.mask.bin'),
        '--batch-size',str(LONG_BATCH_SIZE),
        '--seq-len',str(LONG_SEQ_LEN),
        '--batches','100',
        '--output',str(EVAL_REPORT)
    ],check=True)
    print('Evaluation report:', EVAL_REPORT)

    TASK_EVAL = ART/'evaluation'/'evaluation-tasks.jsonl'
    if TASK_EVAL.is_file():
        TASK_REPORT = ART/'reports'/f'task_eval_{BASE_KEY}.json'
        subprocess.run([
            'python','-m','qmini.cli','eval-tasks',
            '--model-dir',str(BASE/'final_model'),
            '--dataset',str(TASK_EVAL),
            '--output',str(TASK_REPORT)
        ],check=True)
        print('Task evaluation report:', TASK_REPORT)
    else:
        print('Optional task set not found at', TASK_EVAL, '; task-level evaluation was skipped.')


## Stage 3 — optional SFT from Q-Mini's own checkpoint

This stage is separate from pretraining. It uses a user-supplied, licensed instruction/chat dataset in Drive, assistant-only loss masks, matching tokenizer/config fingerprints, and `--init-from` with a Q-Mini-produced checkpoint. It does not download pretrained models. It is disabled by default.


In [ ]:
# Stage 3: optional supervised fine-tuning from a Q-Mini-owned checkpoint.
# Put only licensed, reviewed chat/instruction data in this separate Drive folder.
RUN_OWN_CHECKPOINT_SFT = False
SFT_DATA = ART/'sft_datasets'
SFT_TOKEN_DIR = ART/'sft_tokens_v1'
SFT_STEPS = 50
SFT_BATCH_SIZE = 1
SFT_GRAD_ACCUM = 1
SFT_EVAL_EVERY = 10
SFT_SAVE_EVERY = 10
SFT_EVAL_BATCHES = 8
SFT_LEARNING_RATE = 1e-4
SFT_WEIGHT_DECAY = 0.01
SFT_VALID_FRACTION = 0.02
SFT_NEAR_DUPLICATE_DISTANCE = 0

# Never silently fine-tune the tiny random-init smoke checkpoint.
# Default to the largest completed 0.7M-class Q-Mini pretraining run with
# verified model/tokenizer provenance. Smaller experiments require an override.
SFT_SOURCE_OVERRIDE = None  # optional explicit Q-Mini final_model path
SFT_MINIMUM_SOURCE_PARAMETERS = 500_000

def select_sft_source():
    if SFT_SOURCE_OVERRIDE is not None:
        candidates = [Path(SFT_SOURCE_OVERRIDE).expanduser()]
    else:
        candidates = [
            manifest_path.parent
            for manifest_path in sorted(
                (ART/'runs').glob('base_*/final_model/model_manifest.json')
            )
        ]
    compatible = []
    tokenizer_hash = sha256_file(TOK)
    for candidate in candidates:
        manifest_path = candidate/'model_manifest.json'
        config_path = candidate/'config.json'
        packed_path = candidate/'model.qmini.npz'
        model_tokenizer_path = candidate/'tokenizer.json'
        if not all(path.is_file() for path in (manifest_path, config_path, packed_path, model_tokenizer_path)):
            continue
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        config = QMiniConfig.from_json(config_path)
        parameter_count = config.estimated_parameter_count()
        if manifest.get('format') != 'qmini-final-model-v2':
            continue
        if manifest.get('parameters') != parameter_count:
            continue
        if manifest.get('packed_export_sha256') != sha256_file(packed_path):
            continue
        if manifest.get('tokenizer_sha256') != tokenizer_hash:
            continue
        if sha256_file(model_tokenizer_path) != tokenizer_hash:
            continue
        if SFT_SOURCE_OVERRIDE is None and parameter_count < SFT_MINIMUM_SOURCE_PARAMETERS:
            continue
        compatible.append((parameter_count, candidate))
    if not compatible:
        raise FileNotFoundError(
            'No completed, tokenizer-matched Q-Mini 0.7M-class checkpoint was found. '
            'Run 0.7M pretraining first, or set SFT_SOURCE_OVERRIDE to a completed '
            'Q-Mini final_model directory. The tiny smoke model is never selected silently.'
        )
    selected = max(compatible, key=lambda item: item[0])[1]
    print('Selected Q-Mini SFT source:', selected)
    return selected

if not RUN_OWN_CHECKPOINT_SFT:
    print('Own-checkpoint SFT is disabled. Set RUN_OWN_CHECKPOINT_SFT=True after reviewing the dataset.')
else:
    SFT_MIN_FREE_GPU_GIB = 8.0
    if not torch.cuda.is_available():
        raise RuntimeError(
            'The 0.7M-class SFT stage requires a CUDA GPU. Select a GPU runtime in Colab.'
        )
    sft_free_gpu_bytes, sft_total_gpu_bytes = torch.cuda.mem_get_info(0)
    if sft_free_gpu_bytes < int(SFT_MIN_FREE_GPU_GIB * (1024 ** 3)):
        raise RuntimeError(
            f'SFT blocked: {sft_free_gpu_bytes / 1024**3:.2f} GiB GPU memory is free; '
            f'at least {SFT_MIN_FREE_GPU_GIB:.1f} GiB is required by this launch preflight.'
        )
    print(
        f'SFT GPU: {torch.cuda.get_device_name(0)}; '
        f'{sft_free_gpu_bytes / 1024**3:.2f} GiB free / '
        f'{sft_total_gpu_bytes / 1024**3:.2f} GiB total'
    )
    if not SFT_DATA.is_dir():
        raise FileNotFoundError(
            f'Create a separate instruction dataset folder first: {SFT_DATA}'
        )
    supported_suffixes = {
        '.txt', '.text', '.md', '.markdown', '.json', '.jsonl',
        '.ndjson', '.csv', '.parquet',
    }
    sft_files = sorted(
        file for file in SFT_DATA.rglob('*')
        if file.is_file() and file.suffix.lower() in supported_suffixes
    )
    if not sft_files:
        raise RuntimeError(f'No supported instruction dataset files found under {SFT_DATA}.')
    SFT_SOURCE = select_sft_source()
    if not (SFT_SOURCE/'config.json').is_file() or not (
        SFT_SOURCE/'model.qmini.npz'
    ).is_file():
        raise FileNotFoundError(
            'SFT source must be a completed Q-Mini final_model directory. '
            'No external pretrained checkpoints are accepted.'
        )

    sft_config = QMiniConfig.from_json(SFT_SOURCE/'config.json')
    source_manifest_path = SFT_SOURCE/'model_manifest.json'
    if not source_manifest_path.is_file():
        raise FileNotFoundError('Q-Mini source model_manifest.json is required.')
    source_manifest = json.loads(source_manifest_path.read_text(encoding='utf-8'))
    if source_manifest.get('tokenizer_sha256') != sha256_file(TOK):
        raise RuntimeError('SFT source checkpoint and training tokenizer do not match.')
    SFT_HELDOUT_TASKS = ART/'evaluation'/'sft-heldout-tasks.jsonl'
    SFT_HELDOUT_CONVERSATIONS = ART/'evaluation'/'sft-heldout-conversations.jsonl'
    if not SFT_HELDOUT_TASKS.is_file() or not SFT_HELDOUT_CONVERSATIONS.is_file():
        raise FileNotFoundError(
            'SFT is blocked until both independent held-out task and multi-turn files exist: '
            f'{SFT_HELDOUT_TASKS} and {SFT_HELDOUT_CONVERSATIONS}.'
        )
    training_file_hashes = {sha256_file(file) for file in sft_files}
    SFT_HELDOUT_TASKS_SHA256 = sha256_file(SFT_HELDOUT_TASKS)
    SFT_HELDOUT_CONVERSATIONS_SHA256 = sha256_file(SFT_HELDOUT_CONVERSATIONS)
    heldout_hashes = {SFT_HELDOUT_TASKS_SHA256, SFT_HELDOUT_CONVERSATIONS_SHA256}
    if training_file_hashes.intersection(heldout_hashes):
        raise RuntimeError('SFT train/evaluation files are identical; provide independent holdouts.')
    SFT_SEQ_LEN = min(128, sft_config.max_context)

    # A changed source revision, source weights, SFT corpus or optimizer schedule
    # receives a new Drive run directory; no mismatched contract is overwritten.
    sft_identity = hashlib.sha256()
    sft_identity.update(SOURCE_COMMIT.encode('utf-8'))
    sft_identity.update(IMPLEMENTATION_SHA256.encode('utf-8'))
    sft_identity.update(source_manifest.get('tokenizer_sha256','').encode('utf-8'))
    sft_identity.update(sha256_file(SFT_SOURCE/'config.json').encode('utf-8'))
    sft_identity.update(sha256_file(SFT_SOURCE/'model.qmini.npz').encode('utf-8'))
    sft_identity.update(SFT_HELDOUT_TASKS_SHA256.encode('utf-8'))
    sft_identity.update(SFT_HELDOUT_CONVERSATIONS_SHA256.encode('utf-8'))
    sft_identity.update(json.dumps({
        'config': sft_config.to_dict(),
        'batch_size': SFT_BATCH_SIZE,
        'sequence_length': SFT_SEQ_LEN,
        'gradient_accumulation': SFT_GRAD_ACCUM,
        'learning_rate': SFT_LEARNING_RATE,
        'weight_decay': SFT_WEIGHT_DECAY,
        'eval_every': SFT_EVAL_EVERY,
        'save_every': SFT_SAVE_EVERY,
        'eval_batches': SFT_EVAL_BATCHES,
        'valid_fraction': SFT_VALID_FRACTION,
        'near_duplicate_distance': SFT_NEAR_DUPLICATE_DISTANCE,
    }, sort_keys=True).encode('utf-8'))
    for source_file in sft_files:
        sft_identity.update(source_file.name.encode('utf-8'))
        sft_identity.update(sha256_file(source_file).encode('utf-8'))
    SFT_KEY = sft_identity.hexdigest()[:16]
    TOKENS_SFT = ART/'sft_token_streams'/f'sft_{SFT_KEY}'
    if not (TOKENS_SFT/'dataset.json').is_file():
        if TOKENS_SFT.exists() and any(TOKENS_SFT.iterdir()):
            raise RuntimeError(
                'Incomplete SFT token artifacts already exist. Preserve them and use a new versioned path.'
            )
        prepare_args = [
            'python','-m','qmini.cli','prepare',
            '--data-dir',*[str(file) for file in sft_files],
            '--tokenizer',str(TOK),
            '--output-dir',str(TOKENS_SFT),
            '--valid-fraction',str(SFT_VALID_FRACTION),
            '--near-duplicate-hamming-distance',str(SFT_NEAR_DUPLICATE_DISTANCE),
            '--loss-mask','assistant',
        ]
        subprocess.run(prepare_args, check=True)

    sft_steps_per_token_window = SFT_BATCH_SIZE * SFT_SEQ_LEN * SFT_GRAD_ACCUM
    print('SFT source:', SFT_SOURCE)
    print('SFT parameters:', sft_config.estimated_parameter_count())
    print('SFT data files:', len(sft_files))
    print('SFT token streams:', TOKENS_SFT)
    print('Loss policy: assistant targets only; user/system/tool turns are context.')
    print('Fresh optimizer: --init-from loads model weights only.')

    SFT_BASELINE_REPORT = ART/'reports'/f'sft_baseline_{SFT_KEY}.json'
    SFT_CONVERSATION_BASELINE_REPORT = ART/'reports'/f'sft_conversation_baseline_{SFT_KEY}.json'
    if SFT_HELDOUT_TASKS.is_file():
        subprocess.run([
            'python','-m','qmini.cli','eval-tasks',
            '--model-dir',str(SFT_SOURCE),
            '--dataset',str(SFT_HELDOUT_TASKS),
            '--output',str(SFT_BASELINE_REPORT)
        ],check=True)
    else:
        print('Optional independent SFT held-out task set not found:', SFT_HELDOUT_TASKS)
    if SFT_HELDOUT_CONVERSATIONS.is_file():
        subprocess.run([
            'python','-m','qmini.cli','eval-conversations',
            '--model-dir',str(SFT_SOURCE),
            '--dataset',str(SFT_HELDOUT_CONVERSATIONS),
            '--output',str(SFT_CONVERSATION_BASELINE_REPORT)
        ],check=True)
    else:
        print('Optional held-out multi-turn conversation set not found:', SFT_HELDOUT_CONVERSATIONS)

    SFT_RUN = ART/'runs'/f'sft_{SFT_KEY}'
    sft_args = [
        'python','-m','qmini.cli','train',
        '--config',str(SFT_SOURCE/'config.json'),
        '--train-bin',str(TOKENS_SFT/'train.bin'),
        '--valid-bin',str(TOKENS_SFT/'valid.bin'),
        '--train-mask-bin',str(TOKENS_SFT/'train.mask.bin'),
        '--valid-mask-bin',str(TOKENS_SFT/'valid.mask.bin'),
        '--tokenizer',str(TOK),
        '--output-dir',str(SFT_RUN),
        '--steps',str(SFT_STEPS),
        '--batch-size',str(SFT_BATCH_SIZE),
        '--seq-len',str(SFT_SEQ_LEN),
        '--grad-accum',str(SFT_GRAD_ACCUM),
        '--lr',str(SFT_LEARNING_RATE),
        '--weight-decay',str(SFT_WEIGHT_DECAY),
        '--eval-every',str(SFT_EVAL_EVERY),
        '--save-every',str(SFT_SAVE_EVERY),
        '--eval-batches',str(SFT_EVAL_BATCHES),
        '--init-from',str(SFT_SOURCE/'model.qmini.npz'),
    ]
    sft_checkpoint = SFT_RUN/'latest.pt'
    sft_final_manifest = SFT_RUN/'final_model'/'model_manifest.json'
    if sft_checkpoint.is_file():
        old = torch.load(sft_checkpoint, map_location='cpu', weights_only=True)
        if old.get('config') != sft_config.to_dict():
            raise RuntimeError('Existing SFT checkpoint config differs; it was preserved.')
        checkpoint_step = int(old.get('step', 0))
        if checkpoint_step < SFT_STEPS or not sft_final_manifest.is_file():
            # The run manifest preserves the original --init-from source and hash.
            # Resume must not pass --init-from again; the trainer restores that source contract.
            init_index = sft_args.index('--init-from')
            del sft_args[init_index:init_index + 2]
            if checkpoint_step >= SFT_STEPS:
                sft_args[sft_args.index('--steps') + 1] = str(checkpoint_step + 1)
                sft_args.extend(['--resume','--extend-schedule'])
                print('Completing final export by one resumable optimizer step.')
            else:
                sft_args.append('--resume')
                saved_sft_schedule = int(old.get('schedule_total_steps', SFT_STEPS))
                if SFT_STEPS > saved_sft_schedule:
                    sft_args.append('--extend-schedule')
            subprocess.run(sft_args, check=True)
        else:
            print('Compatible SFT run already complete; checkpoint preserved:', sft_checkpoint)
    else:
        subprocess.run(sft_args, check=True)

    sft_manifest = json.loads((SFT_RUN/'final_model'/'model_manifest.json').read_text())
    if sft_manifest.get('tokenizer_sha256') != sha256_file(TOK):
        raise RuntimeError('SFT output tokenizer hash differs from the training tokenizer.')
    print('Own-checkpoint SFT completed:', SFT_RUN/'final_model')
    print('SFT supervised targets:', sft_manifest.get('supervised_tokens_seen'))
    if SFT_HELDOUT_TASKS.is_file():
        SFT_AFTER_REPORT = ART/'reports'/f'sft_after_{SFT_KEY}.json'
        subprocess.run([
            'python','-m','qmini.cli','eval-tasks',
            '--model-dir',str(SFT_RUN/'final_model'),
            '--dataset',str(SFT_HELDOUT_TASKS),
            '--output',str(SFT_AFTER_REPORT)
        ],check=True)
        baseline = json.loads(SFT_BASELINE_REPORT.read_text(encoding='utf-8'))
        after = json.loads(SFT_AFTER_REPORT.read_text(encoding='utf-8'))
        comparison = {
            'source_model': str(SFT_SOURCE),
            'sft_model': str(SFT_RUN/'final_model'),
            'baseline_task_pass_rate': baseline.get('task_pass_rate'),
            'sft_task_pass_rate': after.get('task_pass_rate'),
            'delta_task_pass_rate': (
                after.get('task_pass_rate',0.0) - baseline.get('task_pass_rate',0.0)
            ),
            'baseline_languages': baseline.get('languages',{}),
            'sft_languages': after.get('languages',{}),
            'note': (
                'Small held-out sets have wide uncertainty; inspect per-case results. '
                'A successful SFT run does not guarantee capability improvement.'
            ),
        }
        comparison_path = ART/'reports'/f'sft_comparison_{SFT_KEY}.json'
        comparison_path.write_text(
            json.dumps(comparison, indent=2, ensure_ascii=False), encoding='utf-8'
        )
        print('SFT held-out task comparison:', comparison_path)

    if SFT_HELDOUT_CONVERSATIONS.is_file():
        SFT_CONVERSATION_AFTER_REPORT = ART/'reports'/f'sft_conversation_after_{SFT_KEY}.json'
        subprocess.run([
            'python','-m','qmini.cli','eval-conversations',
            '--model-dir',str(SFT_RUN/'final_model'),
            '--dataset',str(SFT_HELDOUT_CONVERSATIONS),
            '--output',str(SFT_CONVERSATION_AFTER_REPORT)
        ],check=True)
        baseline_conversations = json.loads(
            SFT_CONVERSATION_BASELINE_REPORT.read_text(encoding='utf-8')
        )
        after_conversations = json.loads(
            SFT_CONVERSATION_AFTER_REPORT.read_text(encoding='utf-8')
        )
        conversation_comparison = {
            'source_model': str(SFT_SOURCE),
            'sft_model': str(SFT_RUN/'final_model'),
            'baseline_conversations': baseline_conversations.get('conversations'),
            'sft_conversations': after_conversations.get('conversations'),
            'baseline_turn_pass_rate': baseline_conversations.get('task_pass_rate'),
            'sft_turn_pass_rate': after_conversations.get('task_pass_rate'),
            'delta_turn_pass_rate': (
                after_conversations.get('task_pass_rate',0.0)
                - baseline_conversations.get('task_pass_rate',0.0)
            ),
            'baseline_languages': baseline_conversations.get('languages',{}),
            'sft_languages': after_conversations.get('languages',{}),
            'baseline_dataset_sha256': baseline_conversations.get('dataset_sha256'),
            'sft_dataset_sha256': after_conversations.get('dataset_sha256'),
            'note': (
                'This measures only the supplied held-out conversations. '
                'Inspect all cases and sample counts before interpreting a difference.'
            ),
        }
        conversation_comparison_path = ART/'reports'/f'sft_conversation_comparison_{SFT_KEY}.json'
        conversation_comparison_path.write_text(
            json.dumps(conversation_comparison, indent=2, ensure_ascii=False),
            encoding='utf-8'
        )
        print('SFT held-out conversation comparison:', conversation_comparison_path)
